In [21]:
!python3 -m pip install virtualenv

In [1]:
!$VIRTUAL_ENV_DIR/python37/bin/activate

In [ ]:
!pip install scikit-learn==0.24.0 causalml==0.12.3 shap==0.36.0 lightgbm==3.2.0 pandas==1.0.5

In [ ]:
!python3.7 -m pip install xgboost

In [1]:
from queryrunner_client import Client
from querybuilder_client import QuerybuilderClient
import pandas as pd
import numpy as np
import math
import logging
import sys
import os
import matplotlib.pylab as plt
import matplotlib.colors
from matplotlib import cm
import scipy.stats as stats
import seaborn as sns
sns.set_style('darkgrid')
from IPython.display import display
import csv
import pickle
import joblib

import sklearn
import statsmodels.api as sm
from sklearn.linear_model import Lasso, LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
# from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor as xgb 
#from query import order_query

pd.set_option('display.max_columns', 500)

logger = logging.getLogger()
logging.getLogger().setLevel(logging.INFO)
qr = Client(user_email=f'{your_email}')
qb = QuerybuilderClient(user_email=f'{your_email}')

In [22]:
!virtualenv --python=python3.7 env

In [23]:
!source ./env/bin/activate

In [24]:
!which python3

In [2]:
### AUUC
# sudo code:
# sort()
# x move 1/n, y up if p==p_hat 
# x move 1/n, y down if p!=p_hat
# calc area

In [3]:
### AUCC
# cost considered case: https://zhuanlan.zhihu.com/p/493811028

In [4]:
# AUUC could convert to loss as NDCG in some condition

In [4]:
!python -version

In [29]:
!install_package_python37.sh add scikit-learn=="0.24.0"

In [ ]:
!install_package_python37.sh add causalml==0.12.3

In [3]:
!python -m pip install  scikit-learn==0.24.0 causalml==0.12.3 shap==0.36.0 lightgbm==3.2.0 pandas==1.0.5

In [14]:
!python -m pip install Cython --install-option="--no-cython-compile"

In [15]:
!python -m pip install --use-pep517 causalml

In [5]:
!python -m pip install pandas

In [3]:
import sys
print(sys.path)

In [2]:
!/dsw/snapshots/snapshot_dsw_default_jupyter/python3/bin/python3.6 -m pip scikit-learn==0.24.0 causalml==0.12.3 shap==0.36.0 lightgbm==3.2.0 pandas==1.0.5

In [4]:
!pip uninstall --yes scikit-learn numpy
!pip install scikit-learn==0.24.0 causalml==0.12.3 shap==0.36.0 lightgbm==3.2.0 pandas==1.0.5

In [12]:
!python3 -m pip install scikit-learn==0.24.0 causalml==0.12.3 shap==0.36.0 lightgbm==3.2.0 pandas==1.0.5

In [6]:
import causalml, shap, lightgbm, pandas

In [6]:
import numpy as np
%matplotlib inline
import pandas as pd
# from causalml.metrics.visualize import get_cumlift
import matplotlib.pyplot as plt
RANDOM_COL = "Random"
plt.style.use("fivethirtyeight")
def get_cumlift(
    df, outcome_col="y", treatment_col="w", treatment_effect_col="tau", random_seed=42
):
    """Get average uplifts of model estimates in cumulative population.

    If the true treatment effect is provided (e.g. in synthetic data), it's calculated
    as the mean of the true treatment effect in each of cumulative population.
    Otherwise, it's calculated as the difference between the mean outcomes of the
    treatment and control groups in each of cumulative population.

    For details, see Section 4.1 of Gutierrez and G{\'e}rardy (2016), `Causal Inference
    and Uplift Modeling: A review of the literature`.

    For the former, `treatment_effect_col` should be provided. For the latter, both
    `outcome_col` and `treatment_col` should be provided.

    Args:
        df (pandas.DataFrame): a data frame with model estimates and actual data as columns
        outcome_col (str, optional): the column name for the actual outcome
        treatment_col (str, optional): the column name for the treatment indicator (0 or 1)
        treatment_effect_col (str, optional): the column name for the true treatment effect
        random_seed (int, optional): random seed for numpy.random.rand()

    Returns:
        (pandas.DataFrame): average uplifts of model estimates in cumulative population
    """

    assert (
        (outcome_col in df.columns)
        and (treatment_col in df.columns)
        or treatment_effect_col in df.columns
    )

    df = df.copy()
    np.random.seed(random_seed)
    random_cols = []
    for i in range(10):
        random_col = "__random_{}__".format(i)
        df[random_col] = np.random.rand(df.shape[0])
        random_cols.append(random_col)

    model_names = [
        x
        for x in df.columns
        if x not in [outcome_col, treatment_col, treatment_effect_col]
    ]

    lift = []
    for i, col in enumerate(model_names):
        sorted_df = df.sort_values(col, ascending=False).reset_index(drop=True)
        sorted_df.index = sorted_df.index + 1

        if treatment_effect_col in sorted_df.columns:
            # When treatment_effect_col is given, use it to calculate the average treatment effects
            # of cumulative population.
            lift.append(sorted_df[treatment_effect_col].cumsum() / sorted_df.index)
        else:
            # When treatment_effect_col is not given, use outcome_col and treatment_col
            # to calculate the average treatment_effects of cumulative population.
            sorted_df["cumsum_tr"] = sorted_df[treatment_col].cumsum()
            sorted_df["cumsum_ct"] = sorted_df.index.values - sorted_df["cumsum_tr"]
            sorted_df["cumsum_y_tr"] = (
                sorted_df[outcome_col] * sorted_df[treatment_col]
            ).cumsum()
            sorted_df["cumsum_y_ct"] = (
                sorted_df[outcome_col] * (1 - sorted_df[treatment_col])
            ).cumsum()

            lift.append(
                sorted_df["cumsum_y_tr"] / sorted_df["cumsum_tr"]
                - sorted_df["cumsum_y_ct"] / sorted_df["cumsum_ct"]
            )

    lift = pd.concat(lift, join="inner", axis=1)
    lift.loc[0] = np.zeros((lift.shape[1],))
    lift = lift.sort_index().interpolate()

    lift.columns = model_names
    lift[RANDOM_COL] = lift[random_cols].mean(axis=1)
    lift.drop(random_cols, axis=1, inplace=True)

    return lift
y_c = np.array(1000*[0])
y_t = y_c.copy()
u = np.array(200*[10]+200*[4]+200*[0]+200*[-2]+200*[-4])
for i in [0,200,400,600,800]:
    y_t[i:i+200] = y_c[i:i+200]+u[i]
r = []
for i in range(0,1000):
    r.append(y_t[i])
    r.append(y_c[i])
    
y = np.array(r)
u = np.concatenate((np.random.normal(10,0.01,[400]),
                    np.random.normal(8,0.01,[400]),
                    np.random.normal(6,0.01,[400]),
                    np.random.normal(4,0.01,[400]),
                    np.random.normal(2,0.01,[400])),axis=0)
metric_dfa = pd.DataFrame([u,
                           y,
                           np.array(1000*[1,0])]).T
metric_dfa.columns=['ite','y','w'] 
lift = get_cumlift(metric_dfa)
lift.plot()
gain = lift.mul(lift.index.values, axis=0)
gain.plot()
print(gain.sum() / gain.shape[0])
print("------------------------")
gain = gain.div(np.abs(gain.iloc[-1, :]))
gain.plot()
print(gain.sum() / gain.shape[0])